# Phase 7b — adaptive-threshold sweep

`ADAPTIVE_THRESHOLD` decides when the decoder stops probing and starts
restricting itself to feedback-consistent words. Phase 7 used **50** on the
reasoning that admissible-set medians go 12,972 → ~211 → ~7 → ~2, so 50 sits
between "still probing" and "resolving". That was a reasonable guess, not a
measurement.

The endpoints disagree:

| threshold | mean | solved |
|---|---:|---:|
| ∞ (always filter) | 3.9472 | **241** |
| 50 | **3.7846** | 239 |

Filtering less wins **faster** but loses **more**. That is a trade-off curve and
this maps it.

## Also swept: the no-model control

At every threshold the same games are played by picking **uniformly at random
among admissible words**. Without that, a threshold that merely narrows the
admissible set would look like a model improvement. Phase 7 measured the two
endpoints as 4.5203 (always filter) and 5.0027 (threshold 50) — the model's
margin was **−0.57** and **−1.22** respectively, so the margin itself varies
with the threshold and needs its own curve.

No training. ~45 min.

---

## How to run

1. **GPU T4 x2**
2. Add Input: the v2 SFT package **and** the Phase 7 adapter dataset
3. Set `PREV_RUN_DIR` to the adapter path
4. Run All, download `wordle_sweep_results.zip`

---
# 1. Setup

In [ ]:
import os, sys, json, time, random, subprocess, importlib, shutil, glob
from collections import Counter

def _pip(p):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)
for mod, pkg in [("transformers", "transformers>=4.44"), ("peft", "peft>=0.11"),
                 ("accelerate", "accelerate>=0.30")]:
    try: importlib.import_module(mod)
    except ImportError: _pip(pkg)
import torch, transformers, peft
import numpy as np

def fix_torchao_peft_conflict():
    try: import peft.import_utils as piu
    except Exception as e: return f"unavailable ({e})"
    try: piu.is_torchao_available(); return "no conflict"
    except ImportError: pass
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                   check=False)
    importlib.invalidate_caches()
    try: piu.is_torchao_available(); return "resolved"
    except ImportError: pass
    piu.is_torchao_available = lambda *a, **k: False
    try:
        import peft.tuners.lora.torchao as t
        t.is_torchao_available = lambda *a, **k: False
    except Exception: pass
    return "patched"
print("torchao:", fix_torchao_peft_conflict())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

# ============================ EDIT THIS =====================================
DATASET_DIR  = None
PREV_RUN_DIR = None      # the Phase 7 adapter dataset
# ============================================================================

MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_NAME = "tree_salet_endgame"
MAX_GUESSES, SEED = 6, 20260817
CONSTRAINED_CHUNK, CONSTRAINED_PRUNE, LENGTH_NORMALISE = 512, True, False

# 0 = never filter (pure legal-word decoding)
# a huge value = always filter (the "consistent" decoder)
THRESHOLDS = [0, 2, 5, 10, 20, 50, 100, 250, 1000, 10**9]
CONTROL_SEEDS = 3          # no-model random-admissible runs per threshold

RESULTS_ROOT = "/kaggle/working/results_sweep"
RESULTS_ZIP  = "/kaggle/working/wordle_sweep_results.zip"
os.makedirs(RESULTS_ROOT, exist_ok=True)

PHASE7 = {"consistent": 3.9472, "adaptive50": 3.7846,
          "control_consistent": 4.5203, "control_adaptive50": 5.0027,
          "classical_entropy": 3.4431, "classical_random": 4.0203,
          "classical_frequency": 3.7927}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("thresholds:", THRESHOLDS)

---
# 2. Dataset and adapter

In [ ]:
REQ = ["sft_package/eval/val_answers.jsonl", "code/wordle_solver.py",
       "code/generate_trajectories.py", "artifacts/feedback_matrix.npy",
       "artifacts/answers.txt", "artifacts/valid_guesses.txt"]

def _has(d):
    try: return all(os.path.exists(os.path.join(d, f)) for f in REQ)
    except OSError: return False

def find_root():
    for root in ([DATASET_DIR] if DATASET_DIR else []) + ["/kaggle/input", "."]:
        if not root or not os.path.isdir(root): continue
        if _has(root): return root
        for dp, dn, _ in os.walk(root):
            dn[:] = [d for d in dn if not d.startswith(".")]
            if _has(dp): return dp
    raise FileNotFoundError("SFT package not found under /kaggle/input")

DATA_ROOT = find_root()
SFT_DIR = os.path.join(DATA_ROOT, "sft_package")
sys.path.insert(0, os.path.join(DATA_ROOT, "code"))
print("dataset:", DATA_ROOT)

from wordle_solver import (load_artifacts, SolverConfig, make_solver, play_game,
                           feedback_code, code_to_pattern, ALL_GREEN)
from generate_trajectories import derive_constraints, render_prompt

BUNDLE = load_artifacts(os.path.join(DATA_ROOT, "artifacts"), mmap=True)
VOCAB = BUNDLE.vocab
LEGAL_LOWER = [g.lower() for g in VOCAB.guesses]
LEGAL_GUESSES = set(w.upper() for w in LEGAL_LOWER)
VAL_ANSWERS = [json.loads(l)["answer"].upper()
               for l in open(os.path.join(SFT_DIR, "eval/val_answers.jsonl"),
                             encoding="utf-8")]
assert len(LEGAL_GUESSES) == 12972 and len(VAL_ANSWERS) == 246

def find_adapter():
    for base in ([PREV_RUN_DIR] if PREV_RUN_DIR else []) + ["/kaggle/input",
                                                            "/kaggle/working"]:
        if not base or not os.path.isdir(base): continue
        for dp, _, fs in os.walk(base):
            if "adapter_config.json" in fs and "checkpoint-" not in dp:
                return dp
    raise FileNotFoundError(
        "No adapter found. Attach the Phase 7 adapter dataset and set "
        "PREV_RUN_DIR, or re-run the Phase 7 notebook to train one.")

ADAPTER_DIR = find_adapter()
print("adapter:", ADAPTER_DIR)

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if TOKENIZER.pad_token is None: TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = "left"

def load_model():
    b = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16,
                                             trust_remote_code=True)
    b.config.use_cache = False
    m = PeftModel.from_pretrained(b, ADAPTER_DIR); m.config.use_cache = True
    return m.eval().cuda()

---
# 3. Decoder

In [ ]:
"""
constrained_decode.py — exact argmax over a fixed legal-word set.

The question this answers is

    "Which of these 12,972 legal Wordle words should I play?"

not

    "Generate arbitrary text and see whether it happens to be a word."

Nothing is filtered after the fact. The model never emits free text at all in
constrained mode: every legal word is *scored*, and the highest-scoring one is
played.

--------------------------------------------------------------------------
Definition of the score
--------------------------------------------------------------------------
For a prompt `p` and a legal word `w`, tokenize exactly as the SFT data did --
`" " + w` -- and append EOS. Then

    score(w) = log P(w | p) = sum_i log P(t_i | p, t_0..t_{i-1})

summed over the word's tokens **and the EOS token**.

Including EOS matters. Without it, a word whose token sequence is a prefix of a
longer word's is scored on a strictly smaller set of constraints and is
systematically over-ranked; with EOS the scores are log-probabilities of
complete strings, so they are directly comparable across different token
lengths. No length normalisation is applied: `score(w)` is exactly the
probability the model assigns to playing `w`, which is the quantity we want to
argmax. (`length_normalise=True` is available for a sensitivity check but is a
heuristic, not the default.)

--------------------------------------------------------------------------
How it is computed
--------------------------------------------------------------------------
Naively this is 12,972 forward passes per turn. Instead:

1. The prompt is run **once** with `use_cache=True`, giving a KV cache and the
   next-token distribution `lp0` over the whole vocabulary.
2. `lp0` already gives the first-token log-probability of every legal word, for
   free -- no forward pass.
3. The remaining tokens are scored by teacher forcing: the prompt's KV cache is
   expanded to a batch of `chunk` rows and the padded `[chunk, L]` word-token
   matrix is pushed through in one forward pass. Causal masking makes the
   right-hand padding inert.

Exact branch-and-bound pruning (`prune=True`, on by default and *exact*):
every per-token log-probability is <= 0, so

    score(w) <= lp0[first_token(w)]

is a valid upper bound. Words are visited in descending order of that bound;
once the best fully-scored word beats the bound of every unvisited word, no
unvisited word can win and the scan stops. The returned argmax is identical to
scoring all 12,972 -- `verify_against_full()` asserts exactly that.

--------------------------------------------------------------------------
What is NOT done here
--------------------------------------------------------------------------
- The candidate-answer list is never consulted. The scorer ranks the full legal
  guess pool; it has no idea which words are still possible.
- The hidden answer is never consulted.
- Repeats are not banned by default (`banned` is opt-in), because banning them
  would be a policy change, not a vocabulary constraint.
"""

import numpy as np
import torch
import torch.nn.functional as F

NEG_INF = float("-inf")


# ---------------------------------------------------------------------------
# KV-cache compatibility
#
# transformers has moved the cache representation twice (legacy tuple ->
# DynamicCache.key_cache/value_cache -> Cache.layers). Rather than pin a
# version, read whichever layout is present and rebuild through the same one.
# `LegalWordScorer.self_test()` verifies the result numerically at runtime, so
# a layout this shim gets wrong fails loudly instead of scoring garbage.
# ---------------------------------------------------------------------------
def _cache_layers(cache):
    if hasattr(cache, "layers"):                        # transformers >= 5
        return [(lyr.keys, lyr.values) for lyr in cache.layers]
    if hasattr(cache, "key_cache"):                     # transformers 4.x
        return list(zip(cache.key_cache, cache.value_cache))
    return [(k, v) for k, v in cache]                   # legacy tuple-of-tuples


def _rebuild_cache(template, layers):
    """Rebuild a cache object of the same kind as `template` from `layers`."""
    if hasattr(template, "layers"):
        import copy
        new = copy.deepcopy(template)
        for lyr, (k, v) in zip(new.layers, layers):
            lyr.keys, lyr.values = k, v
        return new
    if hasattr(template, "key_cache"):
        from transformers.cache_utils import DynamicCache
        new = DynamicCache()
        new.key_cache = [k for k, _ in layers]
        new.value_cache = [v for _, v in layers]
        return new
    return tuple(layers)


def expand_cache(cache, n):
    """Repeat a batch-1 KV cache to batch `n` without recomputing the prompt."""
    out = []
    for k, v in _cache_layers(cache):
        assert k.shape[0] == 1, f"expected batch-1 cache, got {k.shape[0]}"
        out.append((k.expand(n, *k.shape[1:]).contiguous(),
                    v.expand(n, *v.shape[1:]).contiguous()))
    return _rebuild_cache(cache, out)


# ---------------------------------------------------------------------------
class LegalWordScorer:
    """Scores every word in a fixed legal set under a causal LM."""

    def __init__(self, tokenizer, words, device="cuda", chunk=512,
                 length_normalise=False):
        self.tok = tokenizer
        self.words = list(words)
        self.device = device
        self.chunk = chunk
        self.length_normalise = length_normalise
        self.n = len(self.words)

        eos = tokenizer.eos_token_id
        seqs = []
        for w in self.words:
            # EXACTLY the training-time tokenization: " " + WORD, then EOS.
            ids = tokenizer(" " + w, add_special_tokens=False)["input_ids"]
            seqs.append(ids + [eos])
        self.max_len = max(len(s) for s in seqs)

        pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else eos
        tokens = np.full((self.n, self.max_len), pad, dtype=np.int64)
        mask = np.zeros((self.n, self.max_len), dtype=np.float32)
        for i, s in enumerate(seqs):
            tokens[i, :len(s)] = s
            mask[i, :len(s)] = 1.0

        self.tokens = torch.from_numpy(tokens).to(device)      # [N, L]
        self.mask = torch.from_numpy(mask).to(device)          # [N, L]
        self.lengths = torch.from_numpy(mask.sum(1)).to(device)
        self.first_tok = self.tokens[:, 0].clone()             # [N]
        self.index = {w: i for i, w in enumerate(self.words)}

    # -- internals ----------------------------------------------------------
    @torch.no_grad()
    def _prompt_pass(self, model, prompt):
        ids = self.tok(prompt, return_tensors="pt",
                       add_special_tokens=False)["input_ids"].to(self.device)
        out = model(input_ids=ids, use_cache=True)
        lp0 = F.log_softmax(out.logits[0, -1].float(), dim=-1)   # [V]
        return out.past_key_values, lp0, ids.shape[1]

    @torch.no_grad()
    def _score_rows(self, model, cache, lp0, prompt_len, rows):
        """Exact log P(w | prompt) for the word indices in `rows`."""
        idx = rows.to(self.device)
        toks = self.tokens[idx]                                  # [C, L]
        msk = self.mask[idx]
        C, L = toks.shape

        total = lp0[toks[:, 0]] * msk[:, 0]                      # token 0, free
        if L > 1:
            big = expand_cache(cache, C)
            attn = torch.ones(C, prompt_len + L, dtype=torch.long,
                              device=self.device)
            pos = torch.arange(prompt_len, prompt_len + L,
                               device=self.device).unsqueeze(0).expand(C, L)
            out = model(input_ids=toks, past_key_values=big,
                        attention_mask=attn, position_ids=pos, use_cache=False)
            # logits[:, i] predicts token i+1, so positions 1..L-1 read 0..L-2.
            for i in range(1, L):
                lp = F.log_softmax(out.logits[:, i - 1].float(), dim=-1)
                total = total + lp.gather(1, toks[:, i:i + 1]).squeeze(1) * msk[:, i]
            del out, big
        if self.length_normalise:
            total = total / msk.sum(1)
        return total

    # -- public API ---------------------------------------------------------
    @torch.no_grad()
    def score_all(self, model, prompt):
        """Score every legal word. No pruning. Returns a [N] float tensor."""
        cache, lp0, plen = self._prompt_pass(model, prompt)
        out = torch.empty(self.n, dtype=torch.float32, device=self.device)
        for s in range(0, self.n, self.chunk):
            rows = torch.arange(s, min(s + self.chunk, self.n))
            out[rows.to(self.device)] = self._score_rows(
                model, cache, lp0, plen, rows)
        return out

    @torch.no_grad()
    def argmax(self, model, prompt, banned=None, allowed_idx=None, prune=True):
        """Highest-scoring legal word.

        With `prune=True` this is the *same* word `score_all().argmax()` gives;
        the bound is exact, not a heuristic. Returns
        `(word, score, n_chunks_scored, margin)`.

        `margin` is the gap to the runner-up **among words actually scored**.
        With pruning on, unvisited words are known to score below the winner but
        could sit above the runner-up, so `margin` is an upper bound on the true
        margin. It is a diagnostic, never an input to a decision.
        """
        cache, lp0, plen = self._prompt_pass(model, prompt)

        bound = lp0[self.first_tok].clone()                      # [N] upper bd
        # `ban_mask` marks words that must NOT be returned, for any reason.
        # It is applied to the bound (so they sort last and prune early) AND to
        # the scores (so one cannot win from the tail of a live chunk). Masking
        # only the bound is a real bug we shipped once: the excluded word still
        # received a genuine score and could come out on top.
        ban_mask = torch.zeros(self.n, dtype=torch.bool, device=self.device)

        if allowed_idx is not None:
            keep = torch.zeros(self.n, dtype=torch.bool, device=self.device)
            keep[torch.as_tensor(np.asarray(allowed_idx), device=self.device)] = True
            ban_mask |= ~keep
            bound = bound.masked_fill(~keep, NEG_INF)

        if banned:
            hit = [self.index[b] for b in banned if b in self.index]
            if hit:
                ix = torch.tensor(hit, device=self.device)
                bound[ix] = NEG_INF     # sorts them last
                ban_mask[ix] = True     # and removes them from the scores

        # Only ever visit words that could win. Sorting the whole vocabulary and
        # relying on the -inf bound to skip the rest still pushes a full chunk
        # of masked-out words through the model: a 3-word admissible set cost
        # MORE than an unfiltered decision (measured 87s vs 32s on CPU) because
        # both scored 512 rows. Restricting the scan pool to the admissible set
        # makes a small set genuinely cheap, which is the common case once the
        # feedback filter bites.
        if allowed_idx is not None:
            pool = torch.as_tensor(np.asarray(allowed_idx), device=self.device)
            pool = pool[~ban_mask[pool]]
            if pool.numel() == 0:                    # everything banned
                pool = torch.as_tensor(np.asarray(allowed_idx), device=self.device)
        else:
            pool = torch.arange(self.n, device=self.device)
        order = pool[torch.argsort(bound[pool], descending=True)]
        n_scan = int(order.numel())

        best_i, best_s, second = -1, NEG_INF, NEG_INF
        n_chunks = 0
        for s in range(0, n_scan, self.chunk):
            rows = order[s:s + self.chunk]
            if bound[rows[0]] == NEG_INF:
                break                                    # all remaining banned
            if best_s >= bound[rows[0]].item():
                break                # no unvisited word can beat the incumbent
            sc = self._score_rows(model, cache, lp0, plen, rows.cpu())
            # A banned word can still land in the tail of an otherwise-live
            # chunk. Masking the bound alone is not enough -- the score has to
            # be masked too, or the ban is silently ignored.
            sc = sc.masked_fill(ban_mask[rows], NEG_INF)
            n_chunks += 1
            top2 = torch.topk(sc, min(2, sc.numel()))
            if top2.values[0].item() > best_s:
                second = max(second, best_s)
                if top2.values.numel() > 1:
                    second = max(second, top2.values[1].item())
                best_s = top2.values[0].item()
                best_i = rows[top2.indices[0]].item()
            elif top2.values[0].item() > second:
                second = top2.values[0].item()

        margin = None if second == NEG_INF else best_s - second
        return self.words[best_i], best_s, n_chunks, margin

    @torch.no_grad()
    def rank_of(self, scores, words):
        """1-based ranks of `words` under a full `score_all` vector."""
        order = torch.argsort(scores, descending=True)
        pos = torch.empty_like(order)
        pos[order] = torch.arange(order.numel(), device=order.device)
        return {w: int(pos[self.index[w]].item()) + 1
                for w in words if w in self.index}

    # -- feedback-consistent selection (Phase 7) ----------------------------
    @torch.no_grad()
    def select(self, model, prompt, banned=None, allowed_idx=None, prune=True):
        """Pick a word, optionally restricted to an admissible subset.

        `allowed_idx` is an index array into `self.words`. It is intended to
        carry the FEEDBACK-CONSISTENT set: the legal words that would have
        produced exactly the feedback already observed. That set is a pure
        function of the prompt (history + the public word list) -- it never
        touches the answer list -- so restricting to it is the same class of
        move as restricting to legal words.

        Returns a dict, not a tuple, so callers can record why a word was
        chosen. The distinction matters for interpreting results:

            forced       len(allowed) == 1. The filter determined the word.
                         The model contributed nothing and this must NOT be
                         counted as a model decision.
            model_chosen len(allowed) > 1. The model ranked the admissible
                         words and picked one.

        Reporting these together would let a decoder improvement masquerade as
        a model improvement.
        """
        n_allowed = self.n if allowed_idx is None else int(len(allowed_idx))

        if allowed_idx is not None and n_allowed == 0:
            # Cannot happen while the answer is legal and the feedback honest,
            # but degrade to the full pool rather than crash a 4-hour run.
            allowed_idx, n_allowed = None, self.n

        if allowed_idx is not None and n_allowed == 1:
            i = int(allowed_idx[0])
            return {"word": self.words[i], "score": None, "n_chunks": 0,
                    "margin": None, "n_allowed": 1, "forced": True,
                    "model_chosen": False}

        w, s, nch, margin = self.argmax(model, prompt, banned=banned,
                                        allowed_idx=allowed_idx, prune=prune)
        return {"word": w, "score": s, "n_chunks": nch, "margin": margin,
                "n_allowed": n_allowed, "forced": False, "model_chosen": True}

    # -- correctness --------------------------------------------------------
    @torch.no_grad()
    def self_test(self, model, prompt, n_probe=32, atol=None):
        """Check cache-reuse scoring against naive full-sequence scoring.

        This is the guard on the KV-cache shim above. A shim that mishandles a
        transformers version produces essentially random scores -- wrong by
        many nats, with the ordering destroyed. That is the failure this must
        catch, and it is enormous.

        What it must NOT flag is float16 rounding. The two paths run different
        matmul shapes (one sequence of length P+L, versus a batch of L-token
        rows against an expanded P-token cache), so fp16 reduction order
        differs and summed log-probs disagree at the 0.01-0.1 nat level. That
        is arithmetic noise, not a broken cache.

        So the test is two-sided:
          * absolute deviation under a dtype-aware tolerance, and
          * the two score vectors still rank the probe words the same way
            (correlation ~1). A broken shim cannot preserve the ranking.

        Note this discrepancy is a *validation* artifact only. Every one of the
        12,972 words is scored through the same fast path, so the ranking the
        argmax is read off is internally consistent.

        Probe indices are spread evenly across the whole word list, not taken
        from the front: if the prompt cache were mutated in place by the first
        chunk's forward pass, only words in *later* chunks would be wrong, and
        a probe drawn from the front would miss it entirely.
        """
        dtype = next(model.parameters()).dtype
        if atol is None:
            atol = 0.02 if dtype in (torch.float32, torch.float64) else 0.40

        fast = self.score_all(model, prompt)
        p_ids = self.tok(prompt, return_tensors="pt",
                         add_special_tokens=False)["input_ids"].to(self.device)
        n_probe = min(n_probe, self.n)
        probe = [int(round(i * (self.n - 1) / max(n_probe - 1, 1)))
                 for i in range(n_probe)]

        got, ref_all = [], []
        for i in probe:
            ids = self.tokens[i][self.mask[i] > 0].unsqueeze(0)
            full = torch.cat([p_ids, ids], dim=1)
            logits = model(input_ids=full, use_cache=False).logits[0].float()
            lp = F.log_softmax(logits, dim=-1)
            start = p_ids.shape[1] - 1
            ref = sum(lp[start + j, ids[0, j]].item() for j in range(ids.shape[1]))
            if self.length_normalise:
                ref /= ids.shape[1]
            ref_all.append(ref)
            got.append(fast[i].item())

        a = np.asarray(got, dtype=np.float64)
        b = np.asarray(ref_all, dtype=np.float64)
        worst = float(np.max(np.abs(a - b)))
        spread = float(b.max() - b.min())
        corr = (float(np.corrcoef(a, b)[0, 1])
                if a.std() > 1e-9 and b.std() > 1e-9 else 1.0)

        assert corr > 0.999, (
            f"constrained scorer does not preserve the ranking of the naive "
            f"scorer (corr={corr:.6f}). The KV-cache shim in expand_cache() "
            f"does not match this transformers version -- do not trust "
            f"constrained results.")
        assert worst < atol, (
            f"constrained scorer disagrees with naive scoring by {worst:.4f} "
            f"nats (tol {atol} for dtype {dtype}), across a probe score spread "
            f"of {spread:.1f} nats. Ranking is preserved (corr={corr:.6f}), so "
            f"this looks like arithmetic noise rather than a broken cache -- "
            f"but it is larger than expected. Investigate before trusting the "
            f"numbers.")
        return {"max_abs_dev": worst, "corr": corr, "spread": spread,
                "atol": atol, "dtype": str(dtype), "n_probe": n_probe}

    @torch.no_grad()
    def verify_against_full(self, model, prompt):
        """Assert the pruned argmax equals the unpruned argmax."""
        full = self.score_all(model, prompt)
        w_full = self.words[int(full.argmax().item())]
        w_prune, _, n_chunks, _ = self.argmax(model, prompt, prune=True)
        assert w_full == w_prune, (
            f"pruning changed the answer: full={w_full} pruned={w_prune}. "
            f"The branch-and-bound bound is wrong.")
        return w_full, n_chunks


# ---------------------------------------------------------------------------
class HardModeFilter:
    """The legal words consistent with every piece of feedback received.

    A word `w` is admissible iff, for every past guess `g` with observed
    pattern `p`, `feedback_code(g, w) == p`. That is precisely the set a
    hard-mode Wordle player can compute from their own board.

    WHAT THIS IS NOT: the candidate set. Two sets are easy to conflate and the
    difference is the whole justification --

        candidate set   answers consistent with feedback   pool = 2,315 answers
                        -> uses the ANSWER LIST, privileged, never used here
        hard-mode set   legal guesses consistent with it   pool = 12,972 legal
                        -> a pure function of the prompt + the public word list

    The model is never shown this set, its size, the answer, or the answer
    list. It is a decoder-side restriction on which words may be selected,
    exactly like the legal-word constraint.

    Refinement is incremental, so cost is dominated by the first turn and
    collapses immediately after (measured: 12,972 -> ~211 -> ~7 -> ~2).
    """

    def __init__(self, legal_words_lower, feedback_code_fn):
        self.words = list(legal_words_lower)
        self._fb = feedback_code_fn
        self.history = []

    def refine(self, guess, code):
        """Apply one (guess, feedback) pair. `guess` lower-case, `code` int."""
        g = guess.lower()
        self.words = [w for w in self.words if self._fb(g, w) == code]
        self.history.append((g, code))
        return self

    def indices(self, scorer):
        """Index array into `scorer.words` (which are upper-case)."""
        return np.array([scorer.index[w.upper()] for w in self.words
                         if w.upper() in scorer.index], dtype=np.int64)

    def contains(self, word):
        return word.lower() in self.words

    def __len__(self):
        return len(self.words)


LEGAL_WORDS_SORTED = sorted(LEGAL_GUESSES)
SCORER = None
def build_scorer():
    global SCORER
    if SCORER is None:
        SCORER = LegalWordScorer(TOKENIZER, LEGAL_WORDS_SORTED, device="cuda",
                                 chunk=CONSTRAINED_CHUNK,
                                 length_normalise=LENGTH_NORMALISE)
    return SCORER

class GameState:
    __slots__ = ("answer","history","cands","guesses","patterns","remaining",
                 "forced","n_allowed","done","solved","filt","win_forced")
    def __init__(self, answer, n_answers, filt):
        self.answer, self.filt = answer, filt
        self.history = []; self.cands = np.arange(n_answers, dtype=np.int32)
        self.guesses, self.patterns, self.remaining = [], [], []
        self.forced, self.n_allowed = [], []
        self.done = self.solved = False; self.win_forced = None
    def prompt(self, turn):
        h = [(g.lower(), p) for g, p in self.history]
        return render_prompt(turn=turn, history=h,
                             constraints=derive_constraints(h),
                             n_candidates=len(self.cands),
                             guesses_remaining=MAX_GUESSES-turn+1,
                             max_guesses=MAX_GUESSES,
                             candidates=None, show_candidate_count=False)

@torch.no_grad()
def play(model, scorer, answers, threshold):
    """threshold: filter only when |admissible| <= threshold. 0 = never."""
    games = [GameState(a, VOCAB.n_answers,
                       HardModeFilter(LEGAL_LOWER, feedback_code)) for a in answers]
    t0 = time.perf_counter()
    for turn in range(1, MAX_GUESSES+1):
        active = [g for g in games if not g.done]
        if not active: break
        for g in active:
            allowed = g.filt.indices(scorer) if len(g.filt) <= threshold else None
            r = scorer.select(model, g.prompt(turn), allowed_idx=allowed,
                              prune=CONSTRAINED_PRUNE)
            w = r["word"]
            g.forced.append(r["forced"]); g.n_allowed.append(r["n_allowed"])
            c = feedback_code(w.lower(), g.answer.lower())
            g.cands = BUNDLE.fb.filter_indices(g.cands, w.lower(), c)
            g.filt.refine(w.lower(), c)
            assert g.filt.contains(g.answer), "answer left the admissible set"
            g.history.append((w, code_to_pattern(c)))
            g.guesses.append(w); g.patterns.append(code_to_pattern(c))
            g.remaining.append(int(len(g.cands)))
            if c == ALL_GREEN:
                g.solved = g.done = True; g.win_forced = r["forced"]
            elif turn == MAX_GUESSES:
                g.done = True
    return games, time.perf_counter()-t0

def summarize(games, threshold, secs):
    n = len(games)
    sc = [len(g.guesses) if g.solved else MAX_GUESSES+1 for g in games]
    solved = [len(g.guesses) for g in games if g.solved]
    hv = tv = 0
    for g in games:
        seen = []
        for gu, pat in zip(g.guesses, g.patterns):
            tv += 1; greens = {}
            for pg, pp in seen:
                for i,(ch,t) in enumerate(zip(pg,pp)):
                    if t == "G": greens[i] = ch
            if any(gu[i] != ch for i,ch in greens.items()): hv += 1
            seen.append((gu,pat))
    dec = [f for g in games for f in g.forced]
    return {"threshold": threshold, "n_games": n,
            "mean": round(sum(sc)/n, 4), "solved": len(solved),
            "failure_rate_pct": round(100*(n-len(solved))/n, 2),
            "mean_solved_only": round(sum(solved)/len(solved), 4) if solved else None,
            "hard_mode_violation_pct": round(100*hv/max(tv,1), 2),
            "forced_decisions": sum(1 for f in dec if f), "decisions": len(dec),
            "forced_pct": round(100*sum(1 for f in dec if f)/max(len(dec),1), 2),
            "wins_forced": sum(1 for g in games if g.solved and g.win_forced),
            "wins_model": sum(1 for g in games if g.solved and g.win_forced is False),
            "eval_seconds": round(secs, 1)}
print("decoder ready")

---
# 4. The no-model control

Same games, same thresholds, picking uniformly at random among admissible
words. Pure CPU, no GPU, and it runs first so the model curve always has
something to be measured against.

In [ ]:
_cache = {}
def after_first(guess, code):
    k = (guess, code)
    if k not in _cache:
        _cache[k] = [w for w in LEGAL_LOWER if feedback_code(guess, w) == code]
    return _cache[k]

def play_random(ans, rng, threshold, opener="salet"):
    g, allowed = opener, LEGAL_LOWER
    for turn in range(1, MAX_GUESSES+1):
        c = feedback_code(g, ans)
        if c == ALL_GREEN: return turn
        allowed = after_first(g, c) if turn == 1 else \
                  [w for w in allowed if feedback_code(g, w) == c]
        if not allowed: return MAX_GUESSES+1
        g = (allowed[rng.randrange(len(allowed))] if len(allowed) <= threshold
             else LEGAL_LOWER[rng.randrange(len(LEGAL_LOWER))])
    return MAX_GUESSES+1

CONTROL = {}
low = [a.lower() for a in VAL_ANSWERS]
print(f"{'thr':>10}{'mean':>9}{'+/-':>7}{'fail%':>8}")
for thr in THRESHOLDS:
    ms, fs = [], []
    for s in range(CONTROL_SEEDS):
        rng = random.Random(1000+s)
        sc = [play_random(a, rng, thr) for a in low]
        ms.append(sum(sc)/len(sc)); fs.append(100*sum(1 for x in sc if x > MAX_GUESSES)/len(sc))
    CONTROL[thr] = {"mean": round(float(np.mean(ms)), 4),
                    "std": round(float(np.std(ms)), 4),
                    "failure_rate_pct": round(float(np.mean(fs)), 2)}
    print(f"{thr:>10}{CONTROL[thr]['mean']:>9.4f}{CONTROL[thr]['std']:>7.3f}"
          f"{CONTROL[thr]['failure_rate_pct']:>8.1f}")
json.dump(CONTROL, open(os.path.join(RESULTS_ROOT, "control.json"), "w"), indent=2)

---
# 5. The sweep

In [ ]:
SWEEP = {}
path = os.path.join(RESULTS_ROOT, "sweep.json")
if os.path.exists(path):
    SWEEP = {int(k): v for k, v in json.load(open(path)).items()}
    print("resumed:", sorted(SWEEP))

MODEL = None
for thr in THRESHOLDS:
    if thr in SWEEP:
        print(f"threshold {thr}: done, skipping"); continue
    if MODEL is None:
        MODEL = load_model(); SC = build_scorer()
    print(f"--- threshold {thr} ---", flush=True)
    games, secs = play(MODEL, SC, VAL_ANSWERS, thr)
    SWEEP[thr] = summarize(games, thr, secs)
    r = SWEEP[thr]
    print(f"  mean={r['mean']:.4f}  solved={r['solved']}  "
          f"fail={r['failure_rate_pct']:.1f}%  hardviol={r['hard_mode_violation_pct']:.1f}%  "
          f"forced={r['forced_pct']:.1f}%  ({secs:.0f}s)")
    json.dump(SWEEP, open(path, "w"), indent=2)
if MODEL is not None:
    del MODEL; torch.cuda.empty_cache()

---
# 6. The frontier

In [ ]:
import csv
print("=" * 92)
print("ADAPTIVE THRESHOLD SWEEP")
print("=" * 92)
h = (f"{'threshold':>10}{'mean':>9}{'solved':>8}{'fail%':>7}{'hardviol%':>11}"
     f"{'forced%':>9}{'control':>9}{'model gain':>12}")
print(h); print("-"*len(h))
best_mean = best_solved = None
for thr in THRESHOLDS:
    r = SWEEP.get(thr); c = CONTROL.get(thr)
    if not r: continue
    gain = (c["mean"] - r["mean"]) if c else float("nan")
    print(f"{thr:>10}{r['mean']:>9.4f}{r['solved']:>8}{r['failure_rate_pct']:>7.1f}"
          f"{r['hard_mode_violation_pct']:>11.1f}{r['forced_pct']:>9.1f}"
          f"{(c['mean'] if c else float('nan')):>9.4f}{gain:>12.4f}")
    if best_mean is None or r["mean"] < SWEEP[best_mean]["mean"]: best_mean = thr
    if best_solved is None or r["solved"] > SWEEP[best_solved]["solved"]: best_solved = thr

print(f"\nreference: classical entropy {PHASE7['classical_entropy']:.4f}, "
      f"frequency {PHASE7['classical_frequency']:.4f}, "
      f"random {PHASE7['classical_random']:.4f}")
if best_mean is not None:
    b = SWEEP[best_mean]
    print(f"\nBEST MEAN     threshold {best_mean}: {b['mean']:.4f} "
          f"({b['solved']} solved)   Phase 7 @50 was {PHASE7['adaptive50']:.4f}")
if best_solved is not None:
    b = SWEEP[best_solved]
    print(f"BEST SOLVED   threshold {best_solved}: {b['solved']}/246 "
          f"(mean {b['mean']:.4f})")
print("""
Reading it: low threshold = probe more (fewer guesses when it works, more
losses); high threshold = always play a consistent word (safer, slower). The
'model gain' column is the whole point -- it is how much the model beats a
random admissible pick at that threshold, and if it collapses toward zero the
decoder is doing the work, not the model.""")

with open(os.path.join(RESULTS_ROOT, "sweep.csv"), "w", newline="",
          encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["threshold","mean","solved","failure_rate_pct",
                "hard_mode_violation_pct","forced_pct","wins_forced","wins_model",
                "control_mean","model_gain"])
    for thr in THRESHOLDS:
        r = SWEEP.get(thr); c = CONTROL.get(thr)
        if not r: continue
        w.writerow([thr, r["mean"], r["solved"], r["failure_rate_pct"],
                    r["hard_mode_violation_pct"], r["forced_pct"],
                    r["wins_forced"], r["wins_model"],
                    c["mean"] if c else "", round(c["mean"]-r["mean"], 4) if c else ""])

shutil.make_archive(RESULTS_ZIP[:-4], "zip", RESULTS_ROOT)
print(f"\nZIP: {RESULTS_ZIP} ({os.path.getsize(RESULTS_ZIP)/2**20:.2f} MiB)")